In [ ]:
import gmsh

In [ ]:
def generate_mesh_non_periodic(mesh_size_at_embeddings, boundary_vxs, embedding_vxs, embeddings_lines, gui = False, tolerance = 1e-6):
    import gmsh
    import time
    import numpy as np

    start_time = time.time()

    gmsh.initialize()
    gmsh.model.add("mesher_from_embeddings")

    dim = 2

    boundary_point_tag = []
    for vx in boundary_vxs:
        boundary_point_tag.append(gmsh.model.occ.addPoint(*vx, meshSize=mesh_size_at_embeddings))

    line_tags = []
    for i in range(len(boundary_point_tag)):
        line_tag = gmsh.model.occ.addLine(boundary_point_tag[i], boundary_point_tag[(i+1)%len(boundary_point_tag)])
        line_tags.append(line_tag)

    wire_tag = gmsh.model.occ.addWire(line_tags)
    srf_tag = gmsh.model.occ.addPlaneSurface([wire_tag])

    point_index_to_tag = {}
    for idx, vx in enumerate(embedding_vxs):
        point_index_to_tag[idx] = gmsh.model.occ.addPoint(*vx, meshSize=mesh_size_at_embeddings)

    embed_tags = []
    for line in embeddings_lines:
        embed_tags.append(gmsh.model.occ.addLine(point_index_to_tag[line[0]], point_index_to_tag[line[1]]))

    gmsh.model.occ.synchronize()

    for tag in embed_tags:
        gmsh.model.mesh.embed(1, [tag], 2, srf_tag)

    gmsh.model.mesh.generate(dim)

    node_tags, node_coords, node_param = gmsh.model.mesh.getNodes()
    element_tags, elements_node_tags = gmsh.model.mesh.getElementsByType(dim)

    v = node_coords.reshape((len(node_tags),3))
    f = order_faces(v, elements_node_tags.reshape((len(element_tags),3)))

    if(gui): gmsh.fltk.run()

    gmsh.finalize()

    return v, f

In [ ]:
def generate_mesh_non_periodic(mesh_size_at_embeddings, boundary_vxs, embedding_vxs, embeddings_lines, gui = False, tolerance = 1e-6):
    import gmsh
    import time
    import numpy as np

    start_time = time.time()

    gmsh.initialize()
    gmsh.model.add("mesher_from_embeddings")

    dim = 2

    boundary_point_tag = []
    for vx in boundary_vxs:
        boundary_point_tag.append(gmsh.model.occ.addPoint(*vx, meshSize=mesh_size_at_embeddings))

    line_tags = []
    for i in range(len(boundary_point_tag)):
        line_tag = gmsh.model.occ.addLine(boundary_point_tag[i], boundary_point_tag[(i+1)%len(boundary_point_tag)])
        line_tags.append(line_tag)

    wire_tag = gmsh.model.occ.addWire(line_tags)
    srf_tag = gmsh.model.occ.addPlaneSurface([wire_tag])

    point_index_to_tag = {}
    for idx, vx in enumerate(embedding_vxs):
        point_index_to_tag[idx] = gmsh.model.occ.addPoint(*vx, meshSize=mesh_size_at_embeddings)

    embed_tags = []
    for line in embeddings_lines:
        embed_tags.append(gmsh.model.occ.addLine(point_index_to_tag[line[0]], point_index_to_tag[line[1]]))

    gmsh.model.occ.synchronize()

    temp_pts = []
    point_index_to_tag = {}
    for idx,vx in enumerate(embedding_vxs):
        point_index_to_tag[idx] = gmsh.model.occ.addPoint(*vx)
        temp_pts.append(vx)

    embed_tags = []
    embed_edges = []
    for line in embeddings_lines:
        embed_tags.append(gmsh.model.occ.addLine(point_index_to_tag[line[0]], point_index_to_tag[line[1]]))
        embed_edges.append([temp_pts[line[0]], temp_pts[line[1]]])

    gmsh.model.occ.synchronize()
    

    # gmsh.model.mesh.embed(dim-1, embed_tags, dim, srf_tag)

    out, _ = gmsh.model.occ.fragment([(2, srf_tag)], [(1, i) for i in embed_tags])

    gmsh.model.occ.synchronize()

    mesh_size = mesh_size_at_embeddings
    # TODO: these functions might need to be updated to produce uniform mesh.
    gmsh.option.setNumber("Mesh.MeshSizeMin", mesh_size)
    gmsh.option.setNumber("Mesh.MeshSizeMax", mesh_size)
    gmsh.option.setNumber("Mesh.Algorithm", 1)
    gmsh.model.mesh.generate(dim)

    meshing_time = time.time()
    print("Meshing took ", meshing_time - start_time, " seconds")

    node_tags, node_coords, node_param = gmsh.model.mesh.getNodes()
    print("get tag")
    element_tags, elements_node_tags = gmsh.model.mesh.getElementsByType(dim)
    print("get element tag")
    v = node_coords.reshape((len(node_tags),3))
    # f = order_faces(v, elements_node_tags.reshape((len(element_tags),3)))

    # if embed_edges != []:
    #     fusing_data = generate_fusing_data_from_lines(v, embed_edges, tolerance)
    # else:
    #     fusing_data = np.array([False] * len(v))

    # print("Compupting fusing data took ", time.time() - meshing_time, " seconds")

    if(gui): gmsh.fltk.run()

    gmsh.finalize()
    return v
    return v, f, fusing_data

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
boundary_vxs = np.load("../pipeline/spline_generation/boundary.npy")

In [ ]:
embedding_vxs = np.load("../pipeline/spline_generation/sheet_vxs.npy")
embeddings_lines = np.load("../pipeline/spline_generation/concatenated_polylines.npy")

In [ ]:
boundary_edges = [[i + len(embedding_vxs), ((i + 1) % (len(boundary_vxs))) + len(embedding_vxs) ] for i in range(len(boundary_vxs))]

In [ ]:
new_vxs = np.array(list(embedding_vxs) + list(boundary_vxs))
new_edges = np.array(list(embeddings_lines) + list(boundary_edges))

In [ ]:
edge_faces = np.concatenate((new_edges, np.zeros((len(new_edges), 1), dtype = np.int64)), axis = 1)

In [ ]:
import igl

In [ ]:
SV, SVI, SVJ, SF = igl.remove_duplicate_vertices(new_vxs, edge_faces, epsilon = 1e1)

In [ ]:
len(SV), len(new_vxs)

In [ ]:
import matplotlib as mpl

In [ ]:
plt.scatter(boundary_vxs[:, 0], boundary_vxs[:, 1], c = np.arange(len(boundary_vxs)), cmap = mpl.colormaps['Greys'])

In [ ]:
plt.scatter(boundary_vxs[:, 0], boundary_vxs[:, 1])

In [ ]:
plt.scatter(embedding_vxs[:, 0], embedding_vxs[:, 1])

In [ ]:
import sys

In [ ]:
sys.path.append("../../")

In [ ]:
import visualization

In [ ]:
plt.scatter(boundary_vxs[:, 0], boundary_vxs[:, 1], c = np.arange(len(boundary_vxs)), cmap = mpl.colormaps['Greys'])


In [ ]:
visualization.plot_line_segments(embedding_vxs, embeddings_lines[210:280])

In [ ]:
# boundary_vxs = [(0, 0, 0), (1, 0, 0), (1, 1, 0), (0, 1, 0)]
# embedding_vxs = [(0.0, -1, 0), (0., -0.5, 0)]
# embeddings_lines = [(0, 1)]

mesh_size_at_embeddings = 10

v = generate_mesh_non_periodic(mesh_size_at_embeddings, boundary_vxs, embedding_vxs, embeddings_lines, gui=True)